Working with the data

In [1]:
import pandas as pd
import json
import numpy as np
from bs4 import BeautifulSoup
import math
import os

In [2]:
def remove_tags(html):
    soup = BeautifulSoup(html, "html.parser")

    # Find and replace <img> tags with their 'alt' attribute
    for img in soup.find_all('img'):
        alt_text = img.get('alt', '')  # default to empty string if 'alt' is None
        if alt_text:
            # Create a new text node
            new_text = soup.new_string(" " + alt_text + " ")
            img.replace_with(new_text)
        else:
            # Remove the image if no alt text
            img.decompose()

    # Remove all script and style elements
    for tag in soup(['script', 'style']):
        tag.decompose()

    # Extract the text, cleaning up any excessive whitespace
    clean_text = ' '.join(soup.stripped_strings)
    return clean_text


In [3]:

dataset_paths = ['assist2009/skill_builder_data_corrected_collapsed.csv', 
 'assist2012/2012-2013-data-with-predictions-4-final.csv', 
 'assist2017/anonymized_full_release_competition_dataset.csv']
datasets = ['assist2009', 'assist2012', 'assist2017']

pb_df = pd.read_csv('./data_subsets/ProblemBodies_23.csv', low_memory=False)

In [4]:
dataset_paths[0] , datasets[0]

('assist2009/skill_builder_data_corrected_collapsed.csv', 'assist2009')

In [6]:
pb_df.head()

,problem_id,problem_code,assistment_id,problem_set_id,problem_set_type,problem_set_name,curriculum,grade_or_subject,unit,link,...,attempts,correct_count,percent_correct,first_cwa,first_cwa_count,second_cwa,second_cwa_count,third_cwa,third_cwa_count,answer
0,1152985.0,PRA5FZU,780070.0,PSAQKFU,NaN,"8.5 Comparing Linear, Exponential, and Quadrat...",Textbook Curricula,Big Ideas Learning,HS: Purple Algebra I (2014),https://app.assistments.org/find/lv/lesson/328...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Exponential
1,1164390.0,PRA5S4J,789764.0,PSAUDWH,NaN,Variables and Patterns Investigation 1 ACE,Textbook Curricula,Pearson (Prentice-Hall),CMP 3,https://app.assistments.org/find/lv/lesson/329...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1167935.0,PRA5V8Q,792776.0,PSAUK4X,NaN,Variables and Patterns Investigation 3 ACE,Textbook Curricula,Pearson (Prentice-Hall),CMP 3,https://app.assistments.org/find/lv/lesson/329...,...,33.0,21.0,63.6,18,3.0,15,2.0,0.3,1.0,12
3,1152694.0,PRA5FRU,779822.0,PSATNY9,NaN,9.7 Independent and Dependent Events,Textbook Curricula,Glencoe McGraw-Hill,Grade 7: Course 2 (2015),https://app.assistments.org/find/lv/lesson/328...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
4,1217278.0,PRA6768,832131.0,PSAVWDT,NaN,Covering and Surrounding ACE Investigation 4,Textbook Curricula,Pearson (Prentice-Hall),CMP 3,https://app.assistments.org/find/lv/lesson/329...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32


In [7]:
print(pb_df.columns)

Index(['problem_id', 'problem_code', 'assistment_id', 'problem_set_id',
       'problem_set_type', 'problem_set_name', 'curriculum',
       'grade_or_subject', 'unit', 'link', 'skill_codes', 'skill_names',
       'problem_type', 'problem_body', 'attempts', 'correct_count',
       'percent_correct', 'first_cwa', 'first_cwa_count', 'second_cwa',
       'second_cwa_count', 'third_cwa', 'third_cwa_count', 'answer'],
      dtype='object')


In [23]:
pb_df['assistment_id'][1997:2025],pb_df['problem_set_id'][1997:2025]

(1997    1167675.0
 1998     805977.0
 1999    1164522.0
 2000    1164587.0
 2001     806008.0
 2002     806032.0
 2003    1164531.0
 2004     806038.0
 2005     806039.0
 2006     805994.0
 2007     806027.0
 2008    1164529.0
 2009    1164577.0
 2010    1164534.0
 2011    1164630.0
 2012    1164632.0
 2013     783041.0
 2014     783637.0
 2015    1167684.0
 2016    1164540.0
 2017     782672.0
 2018     782670.0
 2019     782657.0
 2020     782673.0
 2021    1164788.0
 2022     783677.0
 2023     887800.0
 2024    1135972.0
 Name: assistment_id, dtype: float64,
 1997    PSABAG8U
 1998     PSAVHVW
 1999    PSABACAA
 2000    PSABACR3
 2001     PSAVHVW
 2002     PSAVHVW
 2003    PSABACAA
 2004     PSAVHVW
 2005     PSAVHVW
 2006     PSAVHVW
 2007     PSAVHVW
 2008    PSABACAA
 2009    PSABACSM
 2010    PSABACAA
 2011    PSABACSM
 2012    PSABACSM
 2013     PSATV76
 2014     PSATX8Q
 2015    PSABAG82
 2016    PSABACR3
 2017     PSATVYP
 2018     PSATVYP
 2019     PSATVPA
 2020     PSATVY

In [24]:
pb_df['problem_set_type'][1997:2025],pb_df['problem_set_name'][1997:2025]

(1997    NaN
 1998    NaN
 1999    NaN
 2000    NaN
 2001    NaN
 2002    NaN
 2003    NaN
 2004    NaN
 2005    NaN
 2006    NaN
 2007    NaN
 2008    NaN
 2009    NaN
 2010    NaN
 2011    NaN
 2012    NaN
 2013    NaN
 2014    NaN
 2015    NaN
 2016    NaN
 2017    NaN
 2018    NaN
 2019    NaN
 2020    NaN
 2021    NaN
 2022    NaN
 2023    NaN
 2024    NaN
 Name: problem_set_type, dtype: object,
 1997                    Lesson 9 Preparing for Assessment
 1998                       Prime Time Investigation 3 ACE
 1999    6-2 Solving Systems Using Substitution: Key Co...
 2000                           Chapter 8 Mid-Chapter Quiz
 2001                       Prime Time Investigation 3 ACE
 2002                       Prime Time Investigation 3 ACE
 2003    6-2 Solving Systems Using Substitution: Key Co...
 2004                       Prime Time Investigation 3 ACE
 2005                       Prime Time Investigation 3 ACE
 2006                       Prime Time Investigation 3 ACE
 2007 

In [5]:
dataset = datasets[0]
dataset_path = dataset_paths[0]

In [25]:
assist_df = pd.read_csv('raw_data/' + dataset_path, encoding = "ISO-8859-1", low_memory=False)
problem_id_column_name = 'problem_id'

In [26]:
assist_df.head()

,Unnamed: 0,order_id,assignment_id,user_id,assistment_id,problem_id,original,correct,attempt_count,ms_first_response,...,hint_count,hint_total,overlap_time,template_id,answer_id,answer_text,first_action,bottom_hint,opportunity,opportunity_original
0,1,33022537,277618,64525,33139,51424,1,1,1,32454,...,0,3,32454,30799,NaN,26,0,NaN,1,1.0
1,2,33022709,277618,64525,33150,51435,1,1,1,4922,...,0,3,4922,30799,NaN,55,0,NaN,2,2.0
2,3,35450204,220674,70363,33159,51444,1,0,2,25390,...,0,3,42000,30799,NaN,88,0,NaN,1,1.0
3,4,35450295,220674,70363,33110,51395,1,1,1,4859,...,0,3,4859,30059,NaN,41,0,NaN,2,2.0
4,5,35450311,220674,70363,33196,51481,1,0,14,19813,...,3,4,124564,30060,NaN,65,0,0.0,3,3.0


In [29]:
len(assist_df)

346860

In [27]:
questions = assist_df[problem_id_column_name].unique()
questions_w_text = pb_df[pb_df['problem_id'].isin(questions)]

In [30]:
len(questions)

26688

In [31]:
questions_w_text

,problem_id,problem_code,assistment_id,problem_set_id,problem_set_type,problem_set_name,curriculum,grade_or_subject,unit,link,...,attempts,correct_count,percent_correct,first_cwa,first_cwa_count,second_cwa,second_cwa_count,third_cwa,third_cwa_count,answer
2190,119299.0,PRACDD6,62585.0,PSAJ78,NaN,(7.SP.C.7a) Probability of a Single Event Skil...,Skill Builders,Grade 7,(SP) Statistics and Probability,https://app.assistments.org/find/lv/lesson/328...,...,61.0,20.0,32.8,"<img style=""width: 237px; height: 53px;"" src=""...",10.0,"<img style=""width: 204px; height: 22px;"" src=""...",9.0,"<img style=""width: 237px; height: 22px;"" src=""...",8.0,"<img style=""width: 246px; height: 57px;"" src=""..."
2654,97581.0,PRAB54Z,55574.0,PSAGGT,NaN,(7.EE.B.4a) Solving Equations (with Combining ...,Skill Builders,Grade 7,(EE) Expressions and Equations,https://app.assistments.org/find/lv/lesson/328...,...,56.0,26.0,46.4,B),12.0,D),8.0,A),8.0,C)
2930,79936.0,PRABSBM,44247.0,PSAGZU,NaN,(8.G.B.7) Pythagorean Theorem - Finding a Miss...,Skill Builders,Grade 8,(G) Geometry,https://app.assistments.org/find/lv/lesson/328...,...,62.0,28.0,45.2,the sum of the two other sides.,13.0,half the base times the height.,8.0,the square of one of the sides.,6.0,the sum of the squares of the two other sides.
2933,76339.0,PRABQWX,42893.0,PSANGJ,NaN,(6.G.A.1) Area of a Trapezoid Skill Builder,Skill Builders,Grade 6,(G) Geometry,https://app.assistments.org/find/lv/lesson/328...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45
2941,89068.0,PRABYHY,50210.0,PSAHSU,NaN,(8.G.A.5) Sum of Interior Angles of Triangles ...,Skill Builders,Grade 8,(G) Geometry,https://app.assistments.org/find/lv/lesson/328...,...,52.0,30.0,57.7,135,3.0,180,2.0,1440,2.0,1080
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
472553,90055.0,PRABZE7,51085.0,PSAHKR,NaN,(7.NS.A.2c) Dividing Integers Skill Builder,Skill Builders,Grade 7,(NS) The Number System,https://app.assistments.org/find/lv/lesson/328...,...,226.0,160.0,70.8,0,18.0,-100,5.0,-10,5.0,-1
472614,90238.0,PRABZJ2,51204.0,PSAHKR,NaN,(7.NS.A.2c) Dividing Integers Skill Builder,Skill Builders,Grade 7,(NS) The Number System,https://app.assistments.org/find/lv/lesson/328...,...,305.0,253.0,83.0,-1,17.0,-2,5.0,1,5.0,-3
472619,90230.0,PRABZJS,51196.0,PSAHKR,NaN,(7.NS.A.2c) Dividing Integers Skill Builder,Skill Builders,Grade 7,(NS) The Number System,https://app.assistments.org/find/lv/lesson/328...,...,203.0,178.0,87.7,-9,2.0,-5,2.0,5,1.0,-6
472622,90174.0,PRABZGY,51140.0,PSAHKR,NaN,(7.NS.A.2c) Dividing Integers Skill Builder,Skill Builders,Grade 7,(NS) The Number System,https://app.assistments.org/find/lv/lesson/328...,...,227.0,179.0,78.9,-3,25.0,27,7.0,1,2.0,3


In [32]:
len(questions_w_text)

7942

In [33]:
print(f'Number of total questions in {dataset} : {len(questions)}')
print(f'Number of questions in {dataset} with text: {len(questions_w_text)}')

Number of total questions in assist2009 : 26688
Number of questions in assist2009 with text: 7942


In [35]:
assist_df_temp = assist_df[assist_df[problem_id_column_name].isin(questions_w_text['problem_id'])]
assist_df_temp.to_csv('../data/' + dataset_path)


In [36]:
with open('../data/'+ dataset + '/keyid2idx.json', 'r') as f:
        keyid2idx_pb_subset = json.load(f)
print(f'Loaded keyid2idx_pb_subset for {dataset}')

Loaded keyid2idx_pb_subset for assist2009


In [37]:
keyid2idx_pb_subset

{'questions': {'54003': 0,
  '53991': 1,
  '54071': 2,
  '54015': 3,
  '53987': 4,
  '53781': 5,
  '54193': 6,
  '53795': 7,
  '54130': 8,
  '54035': 9,
  '49276': 10,
  '53966': 11,
  '54593': 12,
  '90498': 13,
  '89863': 14,
  '89737': 15,
  '60011': 16,
  '71776': 17,
  '90210': 18,
  '54051': 19,
  '53317': 20,
  '53008': 21,
  '54456': 22,
  '66825': 23,
  '66826': 24,
  '66827': 25,
  '51186': 26,
  '66813': 27,
  '84817': 28,
  '53609': 29,
  '53999': 30,
  '86549': 31,
  '53786': 32,
  '49286': 33,
  '49270': 34,
  '49290': 35,
  '49265': 36,
  '49278': 37,
  '85714': 38,
  '84951': 39,
  '84724': 40,
  '84938': 41,
  '84974': 42,
  '84788': 43,
  '85826': 44,
  '86054': 45,
  '61100': 46,
  '87777': 47,
  '86102': 48,
  '88521': 49,
  '87045': 50,
  '54460': 51,
  '54612': 52,
  '58775': 53,
  '60111': 54,
  '109008': 55,
  '108966': 56,
  '108924': 57,
  '109011': 58,
  '108957': 59,
  '87560': 60,
  '87461': 61,
  '87597': 62,
  '87471': 63,
  '87538': 64,
  '87590': 65,
  

In [38]:
problem_ids = list(keyid2idx_pb_subset['questions'].keys())
problem_ids_int = [int(x) for x in problem_ids]
print(f'Number of problems in {dataset}: {len(problem_ids_int)}')

Number of problems in assist2009: 17737


In [40]:
subset_df_html_selected = pb_df[pb_df['problem_id'].isin(problem_ids_int)]